# DenseNet-121 Joint-Guided Native-CAM Rescue Ablation

This validation-only model-selection experiment starts every arm from the same completed `canonical_final_linear_cam` checkpoint. It tests whether weak central-joint supervision can reduce off-joint native-CAM hotspots without materially reducing KL-grading performance.

Run every cell in order. The test split is deliberately not loaded or evaluated. The architecture remains compatible with the production app: DenseNet-121 final features, five 1x1-convolution class maps, and global spatial averaging into five CE logits.

Arms:

1. `ce_control`: CE-only fine-tuning control.
2. `joint_guided_002`: CE plus joint guidance weight 0.02.
3. `joint_guided_005`: CE plus joint guidance weight 0.05.

The notebook selects checkpoints using predictive and localization metrics, exports the worst localization cases for visual review, and never assumes that a plausible CAM is automatically correct.

**Important:** any embedded outputs from an earlier run may reference last_model.pth; run all cells again after the SHA-256 gate passes. Those previous outputs are retained in the timestamped result directory for audit only.


## 1. Environment and imports

In [1]:
import os
import sys
import json
import math
import random
import hashlib
import shutil
import subprocess
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    cohen_kappa_score,
    precision_recall_fscore_support,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True


Mounted at /content/drive
Device: cuda
GPU: Tesla T4


## 2. Controlled configuration

In [2]:
class ExperimentConfig:
    architecture = "canonical_final_linear_cam"
    seed = 42

    # Compare from production-best, never the later last-model file.
    baseline_checkpoint = "/content/drive/MyDrive/Models/densenet121_checkpoints/2026-07-21_15-07-17_633270_UTC_canonical_final_linear_cam/best_model.pth"
    expected_baseline_sha256 = "a8107b9cc7cc9242385f1facfcfc69c251f88697ed2df4dae8a43c1d66729b76"

    dataset_candidates = [
        "/content/Datasets/kaggle_knee_osteoarthritis",
        "/home/viet/Capstone/ml/dataset/kaggle_knee_osteoarthritis",
    ]
    dataset_zip = "/content/drive/MyDrive/Datasets/kaggle_knee_osteoarthritis.zip"
    checkpoint_search_roots = [
        "/content/drive/MyDrive/Models/densenet121_checkpoints",
        "/home/viet/Capstone/ml/checkpoints/densenet121",
    ]
    output_root = (
        "/content/drive/MyDrive/Models/densenet121_joint_guided_ablations"
        if IN_COLAB
        else "/home/viet/Capstone/ml/joint_guided_cam_ablations"
    )

    img_size = 400
    crop_size = 384
    batch_size = 64
    num_workers = 4
    epochs_per_arm = 6
    learning_rate = 5e-6
    weight_decay = 1e-3
    sampler_power = 1.0
    use_amp = True

    rotation_degrees = 5
    brightness_jitter = 0.08
    contrast_jitter = 0.08
    random_erasing_p = 0.10

    # A soft vertical band: enough room for joint margins and osteophytes, but
    # it penalizes strong evidence deep in the tibial shaft or femoral texture.
    joint_center_y = 0.50
    joint_sigma_y = 0.16
    border_cost_weight = 0.20

    cam_cases_per_grade = 50
    gallery_cases_per_arm = 8
    qwk_tolerance = 0.010
    macro_f1_tolerance = 0.015
    grade1_recall_tolerance = 0.030
    shutdown_colab_when_done = True

    arms = [
        {"name": "ce_control", "guidance_weight": 0.00},
        {"name": "joint_guided_002", "guidance_weight": 0.02},
        {"name": "joint_guided_005", "guidance_weight": 0.05},
    ]


def reset_seed(seed=ExperimentConfig.seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


reset_seed()
run_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
run_dir = os.path.join(ExperimentConfig.output_root, f"{run_timestamp}_joint_guided_cam")
os.makedirs(run_dir, exist_ok=False)

if IN_COLAB and os.path.isfile(ExperimentConfig.dataset_zip):
    target = "/content/Datasets"
    if not os.path.isdir(os.path.join(target, "kaggle_knee_osteoarthritis")):
        os.makedirs(target, exist_ok=True)
        subprocess.run(
            ["unzip", "-q", "-o", ExperimentConfig.dataset_zip, "-d", target],
            check=True,
        )

dataset_root = next(
    (path for path in ExperimentConfig.dataset_candidates if os.path.isdir(path)),
    None,
)
if dataset_root is None:
    raise FileNotFoundError(
        "Dataset not found. Update ExperimentConfig.dataset_candidates."
    )
if not os.path.isdir(os.path.join(dataset_root, "val")):
    raise FileNotFoundError("A validation split is required; test fallback is disabled.")

print("Dataset:", dataset_root)
print("Output:", run_dir)
print(json.dumps({
    key: value for key, value in vars(ExperimentConfig).items()
    if not key.startswith("_") and not callable(value)
}, indent=2, default=str))


Dataset: /content/Datasets/kaggle_knee_osteoarthritis
Output: /content/drive/MyDrive/Models/densenet121_joint_guided_ablations/2026-07-22_11-52-13_081467_UTC_joint_guided_cam
{
  "architecture": "canonical_final_linear_cam",
  "seed": 42,
  "baseline_checkpoint": "/content/drive/MyDrive/Models/densenet121_checkpoints/2026-07-21_15-07-17_633270_UTC_canonical_final_linear_cam/best_model.pth",
  "expected_baseline_sha256": "a8107b9cc7cc9242385f1facfcfc69c251f88697ed2df4dae8a43c1d66729b76",
  "dataset_candidates": [
    "/content/Datasets/kaggle_knee_osteoarthritis",
    "/home/viet/Capstone/ml/dataset/kaggle_knee_osteoarthritis"
  ],
  "dataset_zip": "/content/drive/MyDrive/Datasets/kaggle_knee_osteoarthritis.zip",
  "checkpoint_search_roots": [
    "/content/drive/MyDrive/Models/densenet121_checkpoints",
    "/home/viet/Capstone/ml/checkpoints/densenet121"
  ],
  "output_root": "/content/drive/MyDrive/Models/densenet121_joint_guided_ablations",
  "img_size": 400,
  "crop_size": 384,
  "b

## 3. Exact preprocessing, laterality and leakage controls

In [3]:
class SquarePadOpenCV:
    def __call__(self, image):
        height, width = image.shape[:2]
        maximum = max(height, width)
        top = (maximum - height) // 2
        bottom = maximum - height - top
        left = (maximum - width) // 2
        right = maximum - width - left
        return cv2.copyMakeBorder(
            image, top, bottom, left, right,
            borderType=cv2.BORDER_CONSTANT, value=[0, 0, 0]
        )


class OpenCVCLAHE:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, image_rgb):
        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size
        )
        image_lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
        lightness, channel_a, channel_b = cv2.split(image_lab)
        enhanced = clahe.apply(lightness)
        return cv2.cvtColor(
            cv2.merge((enhanced, channel_a, channel_b)), cv2.COLOR_LAB2RGB
        )


def is_right_knee_path(image_path):
    stem = os.path.splitext(os.path.basename(image_path))[0]
    return stem.upper().endswith("R")


def canonicalize_laterality(image, image_path):
    if is_right_knee_path(image_path):
        return np.ascontiguousarray(image[:, ::-1])
    return image


train_transform = transforms.Compose([
    SquarePadOpenCV(),
    OpenCVCLAHE(),
    transforms.ToPILImage(),
    transforms.RandomRotation(ExperimentConfig.rotation_degrees),
    transforms.ColorJitter(
        brightness=ExperimentConfig.brightness_jitter,
        contrast=ExperimentConfig.contrast_jitter,
    ),
    transforms.Resize((ExperimentConfig.img_size, ExperimentConfig.img_size)),
    transforms.RandomCrop(ExperimentConfig.crop_size),
    transforms.ToTensor(),
    transforms.RandomErasing(
        p=ExperimentConfig.random_erasing_p,
        scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0,
    ),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    ),
])

val_transform = transforms.Compose([
    SquarePadOpenCV(),
    OpenCVCLAHE(),
    transforms.ToPILImage(),
    transforms.Resize((ExperimentConfig.img_size, ExperimentConfig.img_size)),
    transforms.CenterCrop(ExperimentConfig.crop_size),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    ),
])


def md5_file(path):
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


class KneeDataset(Dataset):
    def __init__(self, root, split, transform, exclude_hashes=None):
        self.transform = transform
        raw = []
        split_root = os.path.join(root, split)
        for grade in range(5):
            grade_dir = os.path.join(split_root, str(grade))
            if not os.path.isdir(grade_dir):
                raise FileNotFoundError(grade_dir)
            for filename in sorted(os.listdir(grade_dir)):
                if filename.lower().endswith((".png", ".jpg", ".jpeg")):
                    raw.append((os.path.join(grade_dir, filename), grade))

        self.image_paths, self.labels, self.image_hashes = [], [], set()
        duplicates = leaks = 0
        for path, label in raw:
            digest = md5_file(path)
            if exclude_hashes and digest in exclude_hashes:
                leaks += 1
                continue
            if digest in self.image_hashes:
                duplicates += 1
                continue
            self.image_hashes.add(digest)
            self.image_paths.append(path)
            self.labels.append(label)
        print(
            f"{split}: kept={len(self.labels)}, duplicates={duplicates}, "
            f"cross-split leaks={leaks}"
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        path = self.image_paths[index]
        image_bgr = cv2.imread(path)
        if image_bgr is None:
            raise IOError(path)
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        image = canonicalize_laterality(image, path)
        return self.transform(image), self.labels[index], index


train_dataset = KneeDataset(dataset_root, "train", train_transform)
val_dataset = KneeDataset(
    dataset_root, "val", val_transform,
    exclude_hashes=set(train_dataset.image_hashes),
)

class_counts = Counter(train_dataset.labels)
print("Training distribution:", dict(sorted(class_counts.items())))


def make_train_loader():
    weights_by_class = {
        grade: 1.0 / (count ** ExperimentConfig.sampler_power)
        for grade, count in class_counts.items()
    }
    weights = [weights_by_class[label] for label in train_dataset.labels]
    generator = torch.Generator().manual_seed(ExperimentConfig.seed)
    sampler = WeightedRandomSampler(
        weights, num_samples=len(weights), replacement=True, generator=generator
    )
    return DataLoader(
        train_dataset,
        batch_size=ExperimentConfig.batch_size,
        sampler=sampler,
        num_workers=ExperimentConfig.num_workers,
        pin_memory=True,
        persistent_workers=ExperimentConfig.num_workers > 0,
    )


val_loader = DataLoader(
    val_dataset,
    batch_size=ExperimentConfig.batch_size,
    shuffle=False,
    num_workers=ExperimentConfig.num_workers,
    pin_memory=True,
    persistent_workers=ExperimentConfig.num_workers > 0,
)

# Fixed, stratified CAM audit subset shared by every arm and epoch.
audit_indices = []
for grade in range(5):
    grade_indices = [
        index for index, label in enumerate(val_dataset.labels) if label == grade
    ]
    audit_indices.extend(grade_indices[:ExperimentConfig.cam_cases_per_grade])
audit_loader = DataLoader(
    Subset(val_dataset, audit_indices),
    batch_size=ExperimentConfig.batch_size,
    shuffle=False,
    num_workers=ExperimentConfig.num_workers,
    pin_memory=True,
)
print("Fixed CAM audit cases:", len(audit_indices))


train: kept=5778, duplicates=0, cross-split leaks=0
val: kept=826, duplicates=0, cross-split leaks=0
Training distribution: {0: 2286, 1: 1046, 2: 1516, 3: 757, 4: 173}
Fixed CAM audit cases: 227


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 4. App-compatible native-CAM model and joint guidance

In [4]:
class DenseNet121NativeCAM(nn.Module):
    architecture = "canonical_final_linear_cam"

    def __init__(self, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(
            "densenet121", pretrained=pretrained,
            features_only=True, out_indices=(4,),
        )
        channels = self.backbone.feature_info.channels()[0]
        self.class_conv = nn.Conv2d(channels, 5, kernel_size=1)

    def class_maps(self, images):
        return self.class_conv(self.backbone(images)[0])

    @staticmethod
    def logits_from_class_maps(class_maps):
        return class_maps.mean(dim=(2, 3))

    def forward_with_maps(self, images):
        maps = self.class_maps(images)
        return self.logits_from_class_maps(maps), maps

    def forward(self, images):
        return self.forward_with_maps(images)[0]


def joint_guidance_loss(class_maps, labels):
    """Penalize positive true-class evidence far from a soft joint-space band."""
    batch_size, _, height, width = class_maps.shape
    selected = class_maps[torch.arange(batch_size, device=class_maps.device), labels]
    positive = F.softplus(selected)

    y = torch.linspace(0, 1, height, device=class_maps.device).view(1, height, 1)
    x = torch.linspace(0, 1, width, device=class_maps.device).view(1, 1, width)
    soft_joint = torch.exp(
        -0.5 * ((y - ExperimentConfig.joint_center_y) / ExperimentConfig.joint_sigma_y) ** 2
    ).expand(1, height, width)
    border = ((x < 0.08) | (x > 0.92) | (y < 0.08) | (y > 0.92)).float()
    spatial_cost = (1.0 - soft_joint) + ExperimentConfig.border_cost_weight * border

    numerator = (positive * spatial_cost).flatten(1).sum(dim=1)
    denominator = positive.flatten(1).sum(dim=1).clamp_min(1e-8)
    return (numerator / denominator).mean()


def positive_cam(class_maps, class_indices, output_size=None):
    batch = torch.arange(class_maps.size(0), device=class_maps.device)
    cams = F.relu(class_maps[batch, class_indices]).unsqueeze(1)
    if output_size is not None:
        cams = F.interpolate(cams, size=output_size, mode="bilinear", align_corners=False)
    cams = cams.squeeze(1)
    maxima = cams.flatten(1).amax(dim=1).clamp_min(1e-8)
    return cams / maxima[:, None, None]


## 5. Resolve and verify the common baseline checkpoint

In [5]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_checkpoint_file(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def resolve_baseline_checkpoint():
    if ExperimentConfig.baseline_checkpoint:
        path = os.path.abspath(ExperimentConfig.baseline_checkpoint)
        if not os.path.isfile(path):
            raise FileNotFoundError(path)
        return path

    candidates = []
    for root in ExperimentConfig.checkpoint_search_roots:
        if os.path.isdir(root):
            candidates.extend(Path(root).rglob("best_model.pth"))
    candidates = sorted(candidates, key=lambda path: path.stat().st_mtime, reverse=True)

    for candidate in candidates:
        try:
            payload = load_checkpoint_file(str(candidate), map_location="cpu")
            state = payload.get("model_state_dict") if isinstance(payload, dict) else None
            architecture = payload.get("architecture") if isinstance(payload, dict) else None
            if (
                architecture == ExperimentConfig.architecture
                and isinstance(state, dict)
                and "class_conv.weight" in state
            ):
                return str(candidate)
        except Exception:
            continue
    raise FileNotFoundError(
        "No compatible canonical_final_linear_cam checkpoint found. "
        "Set ExperimentConfig.baseline_checkpoint explicitly."
    )


baseline_path = resolve_baseline_checkpoint()
baseline_payload = load_checkpoint_file(baseline_path, map_location="cpu")
baseline_state = baseline_payload["model_state_dict"]

verification_model = DenseNet121NativeCAM(pretrained=False)
verification_model.load_state_dict(baseline_state, strict=True)
del verification_model

baseline_sha256 = sha256_file(baseline_path)
if baseline_sha256 != ExperimentConfig.expected_baseline_sha256:
    raise RuntimeError(
        "Baseline checkpoint SHA-256 mismatch: "
        f"expected {ExperimentConfig.expected_baseline_sha256}, got {baseline_sha256}. "
        "Do not run this comparison from last_model.pth."
    )
print("Baseline checkpoint:", baseline_path)
print("Baseline SHA-256:", baseline_sha256)
print("Baseline epoch:", baseline_payload.get("epoch"))
print("Baseline validation metrics:", {
    key: value for key, value in baseline_payload.get("validation_metrics", {}).items()
    if key not in {"probas", "report"}
})


Baseline checkpoint: /content/drive/MyDrive/Models/densenet121_checkpoints/2026-07-21_15-07-17_633270_UTC_canonical_final_linear_cam/best_model.pth
Baseline SHA-256: a8107b9cc7cc9242385f1facfcfc69c251f88697ed2df4dae8a43c1d66729b76
Baseline epoch: 23
Baseline validation metrics: {'loss': 0.8469922945228097, 'acc': 64.40677966101696, 'qwk': 0.8000991141671313, 'macro_precision': 0.6823201147481281, 'macro_recall': 0.6922156150899282, 'macro_f1': 0.683701628246404, 'grade1_recall': 0.46405228758169936, 'auc': 0.885956313023951, 'ap': 0.715731614924881}


## 6. Predictive and localization evaluation

In [6]:
def predictive_metrics(labels, predictions, probabilities):
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    binary = label_binarize(labels, classes=np.arange(5))
    try:
        macro_auc = roc_auc_score(binary, probabilities, average="macro", multi_class="ovr")
    except ValueError:
        macro_auc = float("nan")
    try:
        macro_ap = average_precision_score(binary, probabilities, average="macro")
    except ValueError:
        macro_ap = float("nan")
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "qwk": float(cohen_kappa_score(labels, predictions, weights="quadratic")),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(macro_f1),
        "grade1_recall": float(recall_score(labels, predictions, labels=[1], average="macro", zero_division=0)),
        "macro_auc": float(macro_auc),
        "macro_ap": float(macro_ap),
    }


@torch.no_grad()
def evaluate_predictive(model, loader):
    model.eval()
    all_labels, all_predictions, all_probabilities = [], [], []
    loss_sum = sample_count = 0
    for images, labels, _ in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=ExperimentConfig.use_amp and device.type == "cuda",
        ):
            logits = model(images)
            loss = F.cross_entropy(logits.float(), labels)
        probabilities = F.softmax(logits.float(), dim=1)
        batch_size = labels.size(0)
        loss_sum += float(loss.item()) * batch_size
        sample_count += batch_size
        all_labels.extend(labels.cpu().tolist())
        all_predictions.extend(probabilities.argmax(dim=1).cpu().tolist())
        all_probabilities.append(probabilities.cpu().numpy())
    probabilities = np.concatenate(all_probabilities)
    metrics = predictive_metrics(
        np.asarray(all_labels), np.asarray(all_predictions), probabilities
    )
    metrics["val_loss"] = loss_sum / max(sample_count, 1)
    return metrics


def individual_cam_metrics(cam):
    height, width = cam.shape
    y, x = np.mgrid[0:height, 0:width]
    y = y / max(height - 1, 1)
    x = x / max(width - 1, 1)
    joint = (y >= 0.32) & (y <= 0.68) & (x >= 0.04) & (x <= 0.96)
    border = (y < 0.08) | (y > 0.92) | (x < 0.08) | (x > 0.92)
    lower_tibia = y > 0.68
    total = float(cam.sum())
    if total <= 1e-8:
        return {
            "joint_energy": 0.0,
            "border_energy": 0.0,
            "lower_tibia_energy": 0.0,
            "peak_inside_joint": 0.0,
            "peak_y_distance": 0.5,
            "empty_cam": 1.0,
        }
    peak_row, peak_col = np.unravel_index(np.argmax(cam), cam.shape)
    peak_y = peak_row / max(height - 1, 1)
    peak_inside = float(joint[peak_row, peak_col])
    return {
        "joint_energy": float(cam[joint].sum() / total),
        "border_energy": float(cam[border].sum() / total),
        "lower_tibia_energy": float(cam[lower_tibia].sum() / total),
        "peak_inside_joint": peak_inside,
        "peak_y_distance": float(abs(peak_y - 0.5)),
        "empty_cam": 0.0,
    }


@torch.no_grad()
def audit_native_cams(model):
    model.eval()
    records = []
    for images, labels, indices in audit_loader:
        images = images.to(device, non_blocking=True)
        labels_device = labels.to(device, non_blocking=True)
        logits, maps = model.forward_with_maps(images)
        probabilities = F.softmax(logits.float(), dim=1)
        predictions = probabilities.argmax(dim=1)
        predicted_cams = positive_cam(maps, predictions).cpu().numpy()
        true_cams = positive_cam(maps, labels_device).cpu().numpy()

        for local_index in range(len(labels)):
            predicted_metrics = individual_cam_metrics(predicted_cams[local_index])
            true_metrics = individual_cam_metrics(true_cams[local_index])
            record = {
                "dataset_index": int(indices[local_index]),
                "path": val_dataset.image_paths[int(indices[local_index])],
                "true_grade": int(labels[local_index]),
                "predicted_grade": int(predictions[local_index]),
                "confidence": float(probabilities[local_index].max().item()),
            }
            record.update({f"pred_{key}": value for key, value in predicted_metrics.items()})
            record.update({f"true_{key}": value for key, value in true_metrics.items()})
            records.append(record)

    frame = pd.DataFrame(records)
    summary = {
        "cam_joint_energy": float(frame["pred_joint_energy"].mean()),
        "cam_border_energy": float(frame["pred_border_energy"].mean()),
        "cam_lower_tibia_energy": float(frame["pred_lower_tibia_energy"].mean()),
        "cam_peak_inside_joint_rate": float(frame["pred_peak_inside_joint"].mean()),
        "cam_peak_y_distance": float(frame["pred_peak_y_distance"].mean()),
        "cam_empty_rate": float(frame["pred_empty_cam"].mean()),
        "true_cam_joint_energy": float(frame["true_joint_energy"].mean()),
        "true_cam_peak_inside_joint_rate": float(frame["true_peak_inside_joint"].mean()),
    }
    return summary, frame


def predictive_score(metrics):
    return (
        0.50 * metrics["qwk"]
        + 0.30 * metrics["macro_f1"]
        + 0.20 * metrics["grade1_recall"]
    )


def localization_score(metrics):
    return (
        0.45 * metrics["cam_peak_inside_joint_rate"]
        + 0.30 * metrics["cam_joint_energy"]
        + 0.15 * (1.0 - metrics["cam_lower_tibia_energy"])
        + 0.10 * (1.0 - metrics["cam_border_energy"])
    )


## 7. Run the three controlled fine-tuning arms

In [7]:
def train_epoch(model, loader, optimizer, scaler, guidance_weight):
    model.train()
    total_loss = total_ce = total_guidance = sample_count = 0
    for images, labels, _ in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=ExperimentConfig.use_amp and device.type == "cuda",
        ):
            logits, maps = model.forward_with_maps(images)
            ce = F.cross_entropy(logits.float(), labels)
            guidance = joint_guidance_loss(maps.float(), labels)
            loss = ce + guidance_weight * guidance
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)
        sample_count += batch_size
        total_loss += float(loss.item()) * batch_size
        total_ce += float(ce.item()) * batch_size
        total_guidance += float(guidance.item()) * batch_size
    return {
        "train_loss": total_loss / sample_count,
        "train_ce": total_ce / sample_count,
        "train_guidance": total_guidance / sample_count,
    }


all_epoch_rows = []
best_rows = []
best_audit_frames = {}

for arm in ExperimentConfig.arms:
    arm_name = arm["name"]
    guidance_weight = float(arm["guidance_weight"])
    print("\n" + "=" * 90)
    print(f"ARM: {arm_name} | guidance_weight={guidance_weight}")
    print("=" * 90)

    reset_seed()
    train_loader = make_train_loader()
    model = DenseNet121NativeCAM(pretrained=False).to(device)
    model.load_state_dict(baseline_state, strict=True)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=ExperimentConfig.learning_rate,
        weight_decay=ExperimentConfig.weight_decay,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=ExperimentConfig.epochs_per_arm, eta_min=1e-7
    )
    scaler = torch.amp.GradScaler(
        "cuda", enabled=ExperimentConfig.use_amp and device.type == "cuda"
    )

    arm_dir = os.path.join(run_dir, arm_name)
    os.makedirs(arm_dir, exist_ok=False)
    best_selection = -float("inf")
    best_row = None

    for epoch in range(1, ExperimentConfig.epochs_per_arm + 1):
        train_metrics = train_epoch(
            model, train_loader, optimizer, scaler, guidance_weight
        )
        val_metrics = evaluate_predictive(model, val_loader)
        cam_metrics, audit_frame = audit_native_cams(model)
        pred_score = predictive_score(val_metrics)
        loc_score = localization_score(cam_metrics)
        selection_score = 0.80 * pred_score + 0.20 * loc_score

        row = {
            "arm": arm_name,
            "guidance_weight": guidance_weight,
            "epoch": epoch,
            "learning_rate": optimizer.param_groups[0]["lr"],
            **train_metrics,
            **val_metrics,
            **cam_metrics,
            "predictive_score": pred_score,
            "localization_score": loc_score,
            "selection_score": selection_score,
        }
        all_epoch_rows.append(row)
        print(
            f"Epoch {epoch}/{ExperimentConfig.epochs_per_arm} | "
            f"QWK={row['qwk']:.4f} F1={row['macro_f1']:.4f} "
            f"G1R={row['grade1_recall']:.4f} | "
            f"peak-in-joint={row['cam_peak_inside_joint_rate']:.3f} "
            f"joint={row['cam_joint_energy']:.3f} "
            f"lower-tibia={row['cam_lower_tibia_energy']:.3f} "
            f"border={row['cam_border_energy']:.3f} | "
            f"selection={selection_score:.4f}"
        )

        if selection_score > best_selection:
            best_selection = selection_score
            best_row = dict(row)
            best_audit_frames[arm_name] = audit_frame.copy()
            checkpoint = {
                "epoch": epoch,
                "architecture": ExperimentConfig.architecture,
                "loss_type": "ce",
                "model_state_dict": model.state_dict(),
                "baseline_checkpoint": baseline_path,
                "baseline_sha256": baseline_sha256,
                "cam_guidance_weight": guidance_weight,
                "validation_metrics": {**val_metrics, **cam_metrics},
                "selection_score": selection_score,
                "training_config": {
                    key: value for key, value in vars(ExperimentConfig).items()
                    if not key.startswith("_") and not callable(value)
                },
            }
            torch.save(checkpoint, os.path.join(arm_dir, "best_model.pth"))
            audit_frame.to_csv(
                os.path.join(arm_dir, "best_cam_audit_cases.csv"), index=False
            )
        scheduler.step()

    best_rows.append(best_row)
    pd.DataFrame([row for row in all_epoch_rows if row["arm"] == arm_name]).to_csv(
        os.path.join(arm_dir, "epoch_metrics.csv"), index=False
    )
    del model, optimizer, scheduler, scaler, train_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

epoch_frame = pd.DataFrame(all_epoch_rows)
best_frame = pd.DataFrame(best_rows)
epoch_frame.to_csv(os.path.join(run_dir, "all_epoch_metrics.csv"), index=False)
best_frame.to_csv(os.path.join(run_dir, "best_arm_metrics.csv"), index=False)
display(best_frame[
    [
        "arm", "epoch", "qwk", "macro_f1", "grade1_recall", "macro_ap",
        "cam_peak_inside_joint_rate", "cam_joint_energy",
        "cam_lower_tibia_energy", "cam_border_energy", "selection_score",
    ]
].sort_values("selection_score", ascending=False))



ARM: ce_control | guidance_weight=0.0


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 1/6 | QWK=0.8047 F1=0.6853 G1R=0.3922 | peak-in-joint=0.974 joint=0.715 lower-tibia=0.166 border=0.120 | selection=0.7222


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 2/6 | QWK=0.7966 F1=0.6849 G1R=0.3791 | peak-in-joint=0.974 joint=0.710 lower-tibia=0.170 border=0.122 | selection=0.7163


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 5/6 | QWK=0.7951 F1=0.6818 G1R=0.3529 | peak-in-joint=0.974 joint=0.713 lower-tibia=0.166 border=0.121 | selection=0.7111


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 6/6 | QWK=0.8055 F1=0.6874 G1R=0.3595 | peak-in-joint=0.974 joint=0.714 lower-tibia=0.165 border=0.121 | selection=0.7178


,arm,epoch,qwk,macro_f1,grade1_recall,macro_ap,cam_peak_inside_joint_rate,cam_joint_energy,cam_lower_tibia_energy,cam_border_energy,selection_score
0,ce_control,4,0.805374,0.692738,0.398693,0.713481,0.964758,0.709569,0.168933,0.121782,0.724096
1,joint_guided_002,4,0.804653,0.691881,0.392157,0.714593,0.964758,0.711728,0.167097,0.121161,0.722754
2,joint_guided_005,1,0.808190,0.683168,0.385621,0.715871,0.977974,0.716129,0.164697,0.119835,0.722583


## 8. Constrained winner selection and CAM galleries

In [8]:
control = best_frame.loc[best_frame["arm"] == "ce_control"].iloc[0]
best_frame["predictive_gate"] = (
    (best_frame["qwk"] >= control["qwk"] - ExperimentConfig.qwk_tolerance)
    & (best_frame["macro_f1"] >= control["macro_f1"] - ExperimentConfig.macro_f1_tolerance)
    & (best_frame["grade1_recall"] >= control["grade1_recall"] - ExperimentConfig.grade1_recall_tolerance)
)

eligible = best_frame[best_frame["predictive_gate"]].copy()
eligible = eligible.sort_values(
    ["localization_score", "predictive_score"], ascending=False
)
winner = eligible.iloc[0]
winner_name = str(winner["arm"])
winner_checkpoint = os.path.join(run_dir, winner_name, "best_model.pth")

print("\nPredictive gate relative to CE control:")
display(best_frame[
    [
        "arm", "predictive_gate", "qwk", "macro_f1", "grade1_recall",
        "cam_peak_inside_joint_rate", "cam_lower_tibia_energy",
        "localization_score", "predictive_score",
    ]
])
print("SELECTED ARM:", winner_name)
print("SELECTED CHECKPOINT:", winner_checkpoint)


MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def display_image_from_tensor(tensor):
    return ((tensor.cpu() * STD + MEAN).clamp(0, 1).permute(1, 2, 0).numpy())


def overlay_cam(image, cam, alpha=0.40):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    return np.clip((1 - alpha) * image + alpha * heatmap, 0, 1)


def save_worst_case_gallery(arm_name):
    checkpoint_path = os.path.join(run_dir, arm_name, "best_model.pth")
    checkpoint = load_checkpoint_file(checkpoint_path, map_location=device)
    model = DenseNet121NativeCAM(pretrained=False).to(device)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()

    frame = best_audit_frames[arm_name].copy()
    frame["failure_score"] = (
        (1.0 - frame["pred_peak_inside_joint"])
        + frame["pred_lower_tibia_energy"]
        + frame["pred_border_energy"]
        + frame["pred_peak_y_distance"]
    )
    worst = frame.sort_values("failure_score", ascending=False).head(
        ExperimentConfig.gallery_cases_per_arm
    )

    figure, axes = plt.subplots(len(worst), 3, figsize=(12, 4 * len(worst)))
    if len(worst) == 1:
        axes = np.asarray([axes])
    for row_index, (_, record) in enumerate(worst.iterrows()):
        dataset_index = int(record["dataset_index"])
        tensor, true_grade, _ = val_dataset[dataset_index]
        batch = tensor.unsqueeze(0).to(device)
        with torch.no_grad():
            logits, maps = model.forward_with_maps(batch)
            probabilities = F.softmax(logits.float(), dim=1)[0]
            predicted_grade = int(probabilities.argmax())
            predicted_cam = positive_cam(
                maps, torch.tensor([predicted_grade], device=device),
                output_size=tensor.shape[-2:],
            )[0].cpu().numpy()
            true_cam = positive_cam(
                maps, torch.tensor([int(true_grade)], device=device),
                output_size=tensor.shape[-2:],
            )[0].cpu().numpy()
        image = display_image_from_tensor(tensor)
        axes[row_index, 0].imshow(image)
        axes[row_index, 0].set_title(
            f"True G{true_grade} | predicted G{predicted_grade} "
            f"({probabilities[predicted_grade]:.3f})"
        )
        axes[row_index, 1].imshow(overlay_cam(image, predicted_cam))
        axes[row_index, 1].set_title(
            f"Pred CAM | peak-in-joint={int(record['pred_peak_inside_joint'])} | "
            f"lower={record['pred_lower_tibia_energy']:.3f}"
        )
        axes[row_index, 2].imshow(overlay_cam(image, true_cam))
        axes[row_index, 2].set_title("True-class CAM")
        for axis in axes[row_index]:
            axis.axis("off")
    figure.suptitle(f"{arm_name}: worst validation CAM cases", fontsize=16)
    figure.tight_layout(rect=[0, 0, 1, 0.99])
    output_path = os.path.join(run_dir, arm_name, "worst_cam_gallery.png")
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(figure)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return output_path


gallery_paths = [save_worst_case_gallery(arm["name"]) for arm in ExperimentConfig.arms]
print("Saved galleries:")
for path in gallery_paths:
    print(" -", path)

selection_summary = {
    "run_timestamp": run_timestamp,
    "baseline_checkpoint": baseline_path,
    "baseline_sha256": baseline_sha256,
    "selected_arm": winner_name,
    "selected_checkpoint": winner_checkpoint,
    "predictive_gate": {
        "qwk_tolerance": ExperimentConfig.qwk_tolerance,
        "macro_f1_tolerance": ExperimentConfig.macro_f1_tolerance,
        "grade1_recall_tolerance": ExperimentConfig.grade1_recall_tolerance,
    },
    "best_arm_metrics": best_frame.to_dict(orient="records"),
}
with open(os.path.join(run_dir, "selection_summary.json"), "w") as handle:
    json.dump(selection_summary, handle, indent=2, default=float)
with open(os.path.join(run_dir, "SELECTED_CHECKPOINT.txt"), "w") as handle:
    handle.write(winner_checkpoint + "\n")

archive_path = shutil.make_archive(run_dir, "zip", root_dir=run_dir)
print("Result directory:", run_dir)
print("Result archive:", archive_path)



Predictive gate relative to CE control:


,arm,predictive_gate,qwk,macro_f1,grade1_recall,cam_peak_inside_joint_rate,cam_lower_tibia_energy,localization_score,predictive_score
0,ce_control,True,0.805374,0.692738,0.398693,0.964758,0.168933,0.859494,0.690247
1,joint_guided_002,True,0.804653,0.691881,0.392157,0.964758,0.167097,0.860479,0.688322
2,joint_guided_005,True,0.808190,0.683168,0.385621,0.977974,0.164697,0.868239,0.686169


SELECTED ARM: joint_guided_005
SELECTED CHECKPOINT: /content/drive/MyDrive/Models/densenet121_joint_guided_ablations/2026-07-22_11-52-13_081467_UTC_joint_guided_cam/joint_guided_005/best_model.pth
Saved galleries:
 - /content/drive/MyDrive/Models/densenet121_joint_guided_ablations/2026-07-22_11-52-13_081467_UTC_joint_guided_cam/ce_control/worst_cam_gallery.png
 - /content/drive/MyDrive/Models/densenet121_joint_guided_ablations/2026-07-22_11-52-13_081467_UTC_joint_guided_cam/joint_guided_002/worst_cam_gallery.png
 - /content/drive/MyDrive/Models/densenet121_joint_guided_ablations/2026-07-22_11-52-13_081467_UTC_joint_guided_cam/joint_guided_005/worst_cam_gallery.png
Result directory: /content/drive/MyDrive/Models/densenet121_joint_guided_ablations/2026-07-22_11-52-13_081467_UTC_joint_guided_cam
Result archive: /content/drive/MyDrive/Models/densenet121_joint_guided_ablations/2026-07-22_11-52-13_081467_UTC_joint_guided_cam.zip


## 9. Interpretation rules

- Do not choose a guided arm if it fails the predictive gate, even if its maps look cleaner.
- A higher peak-inside-joint rate and lower lower-tibia/border energy directly address the failure shown in the problematic CAM.
- Inspect every `worst_cam_gallery.png`. Aggregate means can conceal isolated high-intensity hotspots.
- If `ce_control` wins, the weak geometric prior did not improve this checkpoint safely. Keep the current production model and create reviewed joint annotations before imposing stronger supervision.
- If a guided arm wins, its checkpoint keeps the same production architecture. The app can load it after its validation metadata and configuration are reviewed.

## 10. Release Colab runtime

In [9]:
if IN_COLAB and ExperimentConfig.shutdown_colab_when_done:
    print("All artifacts are saved. Releasing the Colab runtime...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as error:
        print("Automatic runtime release failed:", error)
else:
    print("Runtime release skipped outside Colab or disabled by configuration.")


All artifacts are saved. Releasing the Colab runtime...
